# Importing final dataframe




In [ ]:
# standard libraries
import pandas as pd
import numpy as np
import os
import ast
from datetime import datetime

# Import useful packages from sklearn
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

pd.set_option('notebook_repr_html', True)

## Through Google Colab

In [ ]:
df_injury = pd.read_csv('https://docs.google.com/spreadsheets/d/1TwWSYiSJviNgr16MH7TLhf53hKHkPhKD1gn261zhJOU/export?format=csv&gid=2070040166')
df_injury

In [ ]:
df_outfield = pd.read_csv('https://docs.google.com/spreadsheets/d/17TDSYYY4TEAEHPxTPP87Pn4WKlI2UGChtnRmozz887s/export?format=csv&gid=1112887342', sep=';')
df_outfield

## Through Local Host

In [ ]:
# Get the current working directory
cwd = os.getcwd()
# Get the parent directory
parent_dir = os.path.dirname(cwd)
parent_dir = os.path.dirname(parent_dir)

# Construct the path to your file
file_path = os.path.join(parent_dir, 'raw_data', 'transfermarkt_data_2022.csv')
# Now you can read the file
df_injury = pd.read_csv(file_path)
# Display the first few rows
df_injury

In [ ]:
# Construct the path to your file
file_path = os.path.join(parent_dir, 'raw_data', 'fbref_data_2022.csv')
# Now you can read the file
df_outfield = pd.read_csv(file_path, sep=';')
df_outfield

# Data Cleansing

In this section, we will outline the steps we took to clean the data obtained from Transfermarkt / FBRef and make it usable to train our model.

## Removing Duplicates

In [ ]:
df_outfield = df_outfield.drop_duplicates(subset='player')
df_injury = df_injury.drop_duplicates(subset='name')
print(df_outfield['player'].value_counts())
print(df_injury['name'].value_counts())

### Merging dataframes

We must now merge the Transfermarkt dataframe (containing the injury data) with the FBRef dataframe (containing in-game statistics).

In [ ]:
df_outfield = df_outfield.rename(columns={"player": "name", "team": "club"})
df_injury = pd.merge(df_injury, df_outfield, how='inner', on='name')
df_injury.head()

In [ ]:
# from google.colab import files

# df_injury.to_csv('df.csv')
# files.download('df.csv')

### Unpacking arrays

There are certain columns containing arrays of tuples (historical data) that cannot be used as features.

Therefore, we must unpack the values within these arrays to make usable features for the model.

In [ ]:
# INJURIES COLUMN
df_injury['injuries'] = df_injury['injuries'].apply(lambda arr: ast.literal_eval(arr)) # only run this once to transform into array (currently held as string)
df_injury['n_injuries'] = df_injury['injuries'].apply(lambda arr: len(arr)) # total number of injuries suffered
df_injury['n_severe_injuries'] = df_injury['injuries'].apply(lambda arr: sum([1 for x in arr if int(x[4]) > 60])) # total number of severe injuries suffered (defined as 60 days missed or greater)

### Target column

In the cell below we create our target variable column.

In [ ]:
df_injury['currently_injured'] = df_injury['injuries'].apply(lambda arr: sum([1 for x in arr if x[3] == '-' or datetime.strptime(x[3], "%b %d, %Y") > datetime.now()])) # currently injured players

### Datatype conversions, non-usable features and dummy variables

In [ ]:
# players who have not stepped foot onto the field, will have 0s for 90% of columns
# we will remove these rows from the dataset (total of 24, 1 injured)
df_injury = df_injury[df_injury['minutes_90s'] != 0]

In [ ]:
# df_injury['height'] = df_injury['height'].apply(lambda x: int(str(x).replace(",","")))

#Drop non usable features (categorical/ non numerical)
colums_to_drop = ["name", "club_x", "club_y", "Unnamed: 0", "birth_year", "nationality", "injuries"]
df_injury.drop(columns = colums_to_drop, inplace = True)

# convert age to float64 column
df_injury['age'] = df_injury['age'].apply(lambda row : float(row[0:2])+(float(row[3:6])/365))

# alternate position column
df_injury['alt_position'] = df_injury['position'].apply(lambda row: row.split(',')[1] if ',' in row else '') # might be null instead of ''
df_injury['position'] = df_injury['position'].apply(lambda row: row[0:2])

#dummy variables for role data
df_injury = pd.get_dummies(df_injury, columns=['position', 'alt_position'])
df_injury['position_DF'] = df_injury['position_DF'] + df_injury['alt_position_DF']
df_injury['position_MF'] = df_injury['position_MF'] + df_injury['alt_position_MF']
df_injury['position_FW'] = df_injury['position_FW'] + df_injury['alt_position_FW']

df_injury = df_injury.rename(columns={'aerials_lost,': 'aerials_lost'})
df_injury['aerials_lost'] = df_injury['aerials_lost'].apply(lambda row: float(row.replace(',', '')))
df_injury.head()

#Drop alternate columns
colums_to_drop = ["alt_position_", "alt_position_DF", "alt_position_MF", "alt_position_FW"]
df_injury.drop(columns = colums_to_drop, inplace = True)
df_injury['currently_injured'] = df_injury['currently_injured'].replace(2, 1)


# NEW ADDITION SINCE SUNDAY
#Make columns on per 90 minutes basis
season_tot_cols = list(df_injury.columns[7:43])
season_tot_cols.extend(['cards_yellow', 'cards_red'])

for col in season_tot_cols:
    df_injury[col] = np.where(df_injury['minutes_90s'] != 0, df_injury[col] / df_injury['minutes_90s'], 0)

# engineered?
df_injury['minutes_per_appearance'] = df_injury['minutes'] / df_injury['games']
df_injury['game_starts_percent'] = df_injury['games_starts'] / df_injury['games']

#Drop seasonal cumulative columns
colums_to_drop = ["games", "games_starts", "minutes", "minutes_90s"]
df_injury.drop(columns = colums_to_drop, inplace = True)

In [ ]:
#final data set columns
df_injury.dtypes

# Splitting data into train and test sets

In [ ]:
#create test and train sets
properties = list(df_injury.columns.values)
properties.remove('currently_injured')
X = df_injury[properties]
y = df_injury['currently_injured']


isf = IsolationForest(n_jobs=-1, random_state=1)
isf.fit(X, y)
preds = isf.predict(X)

X['outlier'] = preds
X = X.drop(X[X['outlier'] == -1].index)
X = X.drop('outlier', axis=1)

y = pd.DataFrame(y)
y['outlier'] = preds
y = y.drop(y[y['outlier'] == -1].index)
y = y.drop('outlier', axis=1)
y = y['currently_injured'].squeeze()

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.80, random_state=1)
X_val, X_train, y_val, y_train = train_test_split(X_train, y_train, test_size=0.8, random_state=1)

## Fixing Class Imbalance

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy='auto', n_jobs=-1)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Model Engineering Techniques

## Feature Engineering

### Algorithmic Method using F-score

In [ ]:
# Create a list of F-values for the existing features
feature_F_scores, _ = f_classif(X_train, y_train)

# Iterate through each combination of features
for f1_index, f1 in enumerate(X_train.columns):
  for f2_index, f2 in enumerate(X_train.columns[f1_index + 1:]):
    
    # Multiply the two features to create a new feature
    new_feature = X_train[[f1]].multiply(X_train[f2], axis=0)
    
    # Evaluate F-value of new feature
    F_Score_new, p_value_new = f_classif(new_feature, y_train)
    
    # Evaluate the relative improvement of the new feature
    
    F_score_improvement = F_Score_new[0] / max(feature_F_scores[[f1_index, f2_index]])
    # Print out features that is sufficiently improved 
    
    if F_score_improvement >= 1.5 and F_Score_new[0] >= 250 and p_value_new < 0.05:
        '''
        Note that F_score_improvement >= 1.5 and F_Score_new[0] >= 250 is
        relatively arbitrary, and that other values could be used.
        '''

        print(f'{f1} * {f2} has an F-score of {F_Score_new[0]:.2f}')
        print(f'\tBetter by a factor of {F_score_improvement:.2f} over features in isolation')
        print(f'\tThe result is significant (p = {p_value_new})')
        print('')


### Using Correlation

In [ ]:
# Iterate through each combination of features
for f1_index, f1 in enumerate(X_train.columns):
  for f2_index, f2 in enumerate(X_train.columns[f1_index + 1:]):
    # Multiply the two features to create a new feature
    new_feature = X_train[[f1]].multiply(X_train[f2], axis=0)
    
    new_feature_ser = new_feature.iloc[:,0]
    corr_val = new_feature_ser.corr(y_train)

    # Evaluate F-value of new feature
    F_Score_new, p_value_new = f_classif(new_feature, y_train)
    if corr_val > 0.2 and p_value_new < 0.05:
        print(f'{f1} + {f2} has a correlation of {corr_val:.2f} with the target feature')
        print(f'\tThe result is significant (p = {p_value_new})')


### Final selection

In [ ]:
def new_feature_combos(X):
    X_new = X.copy()
    
    # Correlation of 0.3
    X_new['n_injuries * age'] = X_new.n_injuries * X_new.age

    # Correlation of 0.28
    X_new['n_injuries * touches'] = X_new.n_injuries * X_new.touches
    
    # Correlation of 0.29
    X_new['n_injuries * miscontrols'] = X_new.n_injuries * X_new.miscontrols

    return X_new

X_train = new_feature_combos(X_train)
X_test = new_feature_combos(X_test)
X_val = new_feature_combos(X_val)

## Feature Selection

In [ ]:
selectk = SelectKBest(k=15)
X_new_train = selectk.fit(X_train, y_train)
X_new_train.transform(X_train)
new_cols = X_new_train.get_feature_names_out()

X_train = X_train[new_cols]
X_test = X_test[new_cols]
X_val = X_val[new_cols]

# select_features = SelectKBest(f_classif, k=15)
# select_features = select_features.fit(X_train_scaled, y_train)

# # Get mask of columns that have good features
# feature_mask = select_features.get_support()
# X_train_fe.columns[feature_mask]

# # select features
# X_train_scaled_fs = X_train_scaled.iloc[:, feature_mask]
# X_test_scaled_fs = X_test_scaled.iloc[:, feature_mask]
# X_val_scaled_fs = X_val_scaled.iloc[:, feature_mask]

## Feature Scaling 

In [ ]:
from sklearn.preprocessing import StandardScaler

def standardize_data(X_train, X_out_of_sample, X_val):
    """standardizes all of the data in X_train and X_out_of_sample. The mean and
    standard deviation of each feature (i.e., each column) from the X_train
    data is used to standardize both the X_train and X_out_of sample."""

    X_train_standardized = X_train.copy()
    X_out_of_sample_standardized = X_out_of_sample.copy()
    X_val_standardized = X_val.copy() 

    scaler = StandardScaler()
    
    '''Use scaler to standardize your data. You'll need to fit scaler with your
    training data (use the fit method) and standardize your training and 
    out-of-sample data (use the transform method)'''

    scaler.fit(X_train) 
    X_train_standardized = scaler.transform(X_train_standardized)
    X_out_of_sample_standardized = scaler.transform(X_out_of_sample_standardized)
    X_val_standardized = scaler.transform(X_val_standardized)

    return X_train_standardized, X_out_of_sample_standardized, X_val_standardized, scaler

# Make new data that is scaled
X_train, X_test, X_val, scaler = standardize_data(X_train, X_test, X_val)

# Download Final Train-Test-Val Split

In [ ]:
# Feature data stored in np arrays so must convert back to df.
pd.DataFrame(data=X_train).to_csv('./final_data/X_train.csv')
pd.DataFrame(data=X_test).to_csv('./final_data/X_test.csv')
pd.DataFrame(data=X_val).to_csv('./final_data/X_val.csv')
y_train.to_csv('./final_data/y_train.csv')
y_test.to_csv('./final_data/y_test.csv')
y_val.to_csv('./final_data/y_val.csv')